# Relative vorticity

The vertical component of relative vorticity,
$\zeta = \partial_x v - \partial_y u$.

The original compares an `xgcm` calculation on MOM5's staggered
B-grid against a naive one that converts degrees to metres and
differentiates with `xarray`, and shows the naive one to be wrong.

Neither applies unchanged here. After `regrid`, `uo` and `vo` are
colocated at tracer points, so the staggering that the `xgcm`
method exists to handle is already gone. What survives of the
warning is the metric term, and that is kept: `cell_lengths`
shrinks the zonal cell length by $\cos(\mathrm{lat})$, which is
exactly the correction the naive method dropped.

### Does this diagnostic mean anything at CMIP resolution?

Ask before running it. The original runs on ACCESS-OM2-01 at
0.1°, where the point of the figure is the **mesoscale eddy
field**. Standard CMIP6 oceans are 1°, and they cannot resolve
an eddy anywhere:

| latitude | $L_d$ | 1° | 0.25° | 0.1° |
|---|---|---|---|---|
| 5° | 157 km | 1.4 | 5.8 **resolved** | 14.0 **resolved** |
| 20° | 40 km | 0.4 | 1.5 | 3.6 *permitting* |
| 40° | 21 km | 0.2 | 0.8 | 1.9 |
| 60° | 16 km | 0.1 | 0.6 | 1.4 |

The ratio is the first baroclinic Rossby radius $L_d = c/|f|$
over the grid spacing, taking the **coarser** of the two
directions, since an eddy has to be resolved in both — and a
uniform lat/lon grid is strongly anisotropic at high latitude,
where $\Delta x$ shrinks with $\cos(\mathrm{lat})$ and
$\Delta y$ does not. Eddies need roughly 2 points per
deformation radius to exist and 4 to be represented.

**A 1° grid does not reach 2 anywhere** — its maximum is 1.88,
in the deep tropics. Those models parameterise eddies
(Gent–McWilliams) rather than resolving them.

Worth noting how demanding this is: even 0.1°, the resolution
the original uses, only reaches 2 within about ±38° of the
equator. "Eddy-permitting" is a generous label for such a model
in the Southern Ocean.

So at 1° this notebook does not show you eddies. It shows the
**shear of the resolved large-scale currents** — western
boundary currents, the ACC fronts, the equatorial current
system — with $|\mathrm{Ro}| \sim 0.01$ rather than
$O(1)$. That is a real field and a legitimate thing to compare
between models, but it is not what the original is about, and
reading it as an eddy field would be wrong.

Two configurations where it does earn its place:

- **HighResMIP**, where ocean grids reach 1/4° to 1/12°
  (HadGEM3-GC31-HH, CMCC-CM2-VHR4, EC-Earth3P-HR). Check
  availability of `uo`/`vo` per model.
- **A resolution pair from one model family** — the change in
  $\zeta/f$ between a 1° and a 1/4° member is itself a
  measure of what resolution buys you.

The cell below computes the ratio for whatever grid you loaded
and says which case you are in.

---

Translated from the COSIMA recipe
[`Relative_Vorticity.ipynb`](https://github.com/COSIMA/cosima-recipes/blob/main/03-Advanced-Recipes/Relative_Vorticity.ipynb)
onto CMORised CMIP6 data, using ESMValCore's `Dataset` and
preprocessors in place of the ACCESS-NRI intake catalog.

This notebook is standalone: it needs only `esmvalcore`,
`iris`, `numpy`, `matplotlib`, `cartopy` and `gsw`. The
equivalent ESMValTool recipe is
`recipes/recipe_cosima_*.yml`.

In [ ]:
import numpy as np
import iris
import iris.analysis.cartography
import iris.plot as iplt
import matplotlib.pyplot as plt

from esmvalcore.dataset import Dataset
from esmvalcore.preprocessor import climate_statistics, regrid

## Start a dask cluster

Every COSIMA recipe opens with one, and these need one for the same
reason, though it is worth being precise about *which* part it speeds up.

**What the cluster does accelerate.** Everything from `Dataset.load()`
through the ESMValCore preprocessors — reading multi-decade files,
`regrid`, `climate_statistics`, `extract_region` — is lazy and chunked.
That is the expensive part of these notebooks and it is what the workers
parallelise. Without a client, dask falls back to its threaded scheduler;
ESMValCore itself warns that the distributed scheduler is preferable.

**What it does not.** Once the preprocessed cube is reduced, the
diagnostic arithmetic realises it into a plain masked array and proceeds
eagerly in numpy. That is deliberate: contour tracing, per-column
remapping and density binning are iterative and index-based, so they do
not express well as array graphs and would be slower, not faster, spread
over workers. Realising *after* the time-mean is what keeps that
affordable — realising a 20-year daily 3-D field before reducing it would
not be.

So: size the cluster for the load-and-preprocess stage. One thread per
worker, as the COSIMA recipes use — with chunks this large, per-worker
memory binds well before thread count does.

On ARE, `Client(threads_per_worker=1)` picks up the cores you requested
for the session. Pass `n_workers=` and `memory_limit=` to be explicit.

**Running the equivalent recipe instead?** Do not start a client there.
ESMValTool manages dask itself, and it defaults to the threaded
scheduler; switch it on in your ESMValTool configuration:

```yaml
dask:
  use: local_distributed
```

In [ ]:
import logging
import os

from dask.distributed import Client

# distributed logs every worker registration and teardown at INFO,
# which is hundreds of lines per notebook and buries everything the
# diagnostic prints. Warnings and errors still come through.
for _name in ("distributed", "distributed.scheduler", "distributed.nanny",
              "distributed.worker", "distributed.core",
              "distributed.batched", "distributed.utils_perf"):
    logging.getLogger(_name).setLevel(logging.WARNING)

# Client.current() reuses a cluster if this cell is re-run, instead of
# quietly starting a second one alongside the first.
try:
    worker_count = max(1, int(os.environ.get("PBS_NCPUS", "1")))
except ValueError:
    worker_count = 1
    print("PBS_NCPUS is not an integer; using one worker")
try:
    client = Client.current()
except ValueError:
    try:
        # Separate worker processes: real memory isolation, and what
        # you want on ARE or any batch node.
        client = Client(n_workers=worker_count, threads_per_worker=1)
    except (RuntimeError, OSError) as error:
        # Some containers and locked-down environments cannot spawn
        # worker subprocesses. In-process workers still give a
        # distributed scheduler and a dashboard, just no isolation.
        print(f'process workers unavailable ({error}); '
              f'using in-process workers instead')
        client = Client(n_workers=worker_count, threads_per_worker=1,
                        processes=False)

active_workers = len(client.scheduler_info()['workers'])
print('dashboard:', client.dashboard_link)
print(f'workers: {active_workers} active (requested {worker_count})')
client

## Load the data

In [ ]:
common = dict(project='CMIP6', mip='Omon', exp='historical',
              ensemble='r1i1p1f1', grid='gn',
              timerange='2000/2009')

uo = Dataset(short_name='uo', dataset='ACCESS-CM2', **common)
vo = uo.copy(short_name='vo')

def prepare(dataset):
    cube = climate_statistics(dataset.load(), operator='mean',
                              period='full')
    return regrid(cube, target_grid='0.5x0.5', scheme='linear')

uo_cube, vo_cube = prepare(uo), prepare(vo)

## Helpers

In [ ]:
# The helpers below are lifted verbatim from
# cosima_cmip/cosima_common.py by notebooks/build_notebooks.py, so
# this notebook runs exactly the same code as the ESMValTool
# diagnostic script. Edit them there, then rebuild.
import contextlib
import warnings

try:  # TEOS-10, needed by the density helpers
    import gsw
except ImportError:
    gsw = None


EARTH_RADIUS = 6371000.0  # m, matches iris.analysis.cartography


FILL_VALUE_THRESHOLD = 1.0e19


def masked_data(cube, dtype=float):
    """Realised, fully masked data of a cube.

    Always go through this rather than ``cube.core_data()``.  The latter
    returns a *lazy* dask array, on which ``numpy.ma`` operations are
    silently no-ops: the mask is not applied, and fill values flow
    straight into the arithmetic.  This realises the data, applies any
    mask the file carries, and additionally masks non-finite values and
    unconverted fill values, which real CMOR files do contain.
    """
    data = np.ma.masked_invalid(np.ma.asarray(cube.data, dtype=dtype))
    return np.ma.masked_where(
        np.abs(np.ma.filled(data, 0.0)) >= FILL_VALUE_THRESHOLD, data)


def _coord_points(cube, name):
    """1D points for ``name``, taking column/row 0 of a 2D aux coord."""
    coord = cube.coord(name)
    if coord.ndim == 1:
        return coord.points
    points = coord.points
    # On a curvilinear grid latitude varies down a column and longitude
    # along a row; take the slice that actually varies.
    if name == "latitude":
        return points[:, 0]
    return points[0, :]


def lat_1d(cube):
    """Representative 1D latitude, also for 2D (curvilinear) coordinates."""
    return _coord_points(cube, "latitude")


def lon_1d(cube):
    """Representative 1D longitude, also for 2D coordinates."""
    return _coord_points(cube, "longitude")


def depth_coord_name(cube):
    """Name of the vertical coordinate, or None for a 2D field."""
    for name in ("depth", "ocean_sigma_z", "olevel", "lev"):
        if cube.coords(name):
            return name
    for coord in cube.coords(dim_coords=True):
        if coord.units.is_convertible("m") and coord.name() != "latitude":
            return coord.name()
    return None


def cell_lengths(cube):
    """Zonal and meridional cell lengths in metres.

    Returns ``(dx, dy)`` broadcast to the horizontal shape of ``cube``.
    ``dx`` shrinks with the cosine of latitude, ``dy`` does not.
    """
    lat = lat_1d(cube)
    lon = lon_1d(cube)
    dlon = np.gradient(lon) * np.pi / 180.0
    dlat = np.gradient(lat) * np.pi / 180.0
    coslat = np.cos(np.deg2rad(lat))
    dx = EARTH_RADIUS * coslat[:, None] * dlon[None, :]
    dy = EARTH_RADIUS * np.broadcast_to(dlat[:, None], dx.shape)
    return dx, np.array(dy)


def horizontal_gradient(field, cube, periodic=True):
    """Central-difference gradient of ``field`` in m-1 units.

    ``field`` is a 2D masked array on the horizontal grid of ``cube``.
    Returns ``(d/dx, d/dy)``.  The zonal direction wraps when
    ``periodic`` is set, which is correct for a global CMOR grid.
    """
    dxm, dym = cell_lengths(cube)
    data = np.ma.masked_invalid(np.ma.asarray(field, dtype=float))
    filled = data.filled(np.nan)

    if periodic:
        padded = np.concatenate(
            [filled[:, -1:], filled, filled[:, :1]], axis=1)
        dfdx = (padded[:, 2:] - padded[:, :-2]) / (2.0 * dxm)
    else:
        dfdx = np.full_like(filled, np.nan)
        dfdx[:, 1:-1] = (filled[:, 2:] - filled[:, :-2]) / (2.0 * dxm[:, 1:-1])

    dfdy = np.full_like(filled, np.nan)
    dfdy[1:-1, :] = (filled[2:, :] - filled[:-2, :]) / (2.0 * dym[1:-1, :])

    return np.ma.masked_invalid(dfdx), np.ma.masked_invalid(dfdy)


def relative_vorticity(u_field, v_field, cube):
    """Vertical component of relative vorticity, ``dv/dx - du/dy``.

    This is the regular-grid equivalent of the ``xgcm`` curl in
    ``Relative_Vorticity.ipynb``.  Both velocity components must already
    sit on the same (tracer) points, which is what ``regrid`` delivers.
    """
    dvdx, _ = horizontal_gradient(v_field, cube)
    _, dudy = horizontal_gradient(u_field, cube)
    return dvdx - dudy


def coriolis(lat):
    """Coriolis parameter in s-1."""
    if gsw is not None:
        return np.asarray(gsw.f(np.asarray(lat, dtype=float)))
    omega = 7.292115e-5
    return 2.0 * omega * np.sin(np.deg2rad(np.asarray(lat, dtype=float)))


GRAVITY_WAVE_SPEED = 2.0


EDDY_PERMITTING_RATIO = 2.0


EDDY_RESOLVING_RATIO = 4.0


def rossby_radius(lat, wave_speed=GRAVITY_WAVE_SPEED):
    """First baroclinic Rossby deformation radius in metres.

    ``L_d = c / |f|``, the length scale of mesoscale eddies.  It
    diverges at the equator, where the equatorial deformation radius
    ``sqrt(c / 2 beta)`` takes over, so the value is capped there.
    """
    lat = np.asarray(lat, dtype=float)
    fcor = np.abs(coriolis(lat))
    beta = 2.0 * 7.292115e-5 / EARTH_RADIUS
    equatorial = np.sqrt(wave_speed / (2.0 * beta))
    with np.errstate(divide="ignore", invalid="ignore"):
        radius = wave_speed / fcor
    return np.minimum(np.where(np.isfinite(radius), radius, equatorial),
                      equatorial)


def eddy_resolution_ratio(cube, wave_speed=GRAVITY_WAVE_SPEED):
    """Deformation radius divided by grid spacing, per latitude.

    This is the question a vorticity or EKE diagnostic has to ask
    before it means anything: *can this grid support an eddy here?*
    Below :data:`EDDY_PERMITTING_RATIO` it cannot, and the vorticity
    field is the shear of the resolved large-scale currents rather
    than anything mesoscale.

    A 1-degree ocean -- what most CMIP6 models run -- does not reach 2
    anywhere outside the deep tropics.
    """
    lat = lat_1d(cube)
    dxm, dym = cell_lengths(cube)
    # The COARSER direction binds: an eddy has to be resolved in both.
    # Taking the finer one flatters a grid that is anisotropic at high
    # latitude, which a uniform lat/lon grid always is -- dx shrinks
    # with cos(lat) while dy does not.
    spacing = np.maximum(np.abs(dxm).max(axis=1), np.abs(dym).max(axis=1))
    return lat, rossby_radius(lat, wave_speed) / spacing

## Compute

In [ ]:
# Can this grid support an eddy? Answer first.
lat_ratio, ratio = eddy_resolution_ratio(uo_cube)
if ratio.max() < EDDY_PERMITTING_RATIO:
    print(f'grid resolves NO deformation radius (max L_d/dx = '
          f'{ratio.max():.2f}, need {EDDY_PERMITTING_RATIO:.0f}).')
    print('-> what follows is large-scale current shear, '
          'NOT mesoscale eddies.')
else:
    band = lat_ratio[ratio >= EDDY_PERMITTING_RATIO]
    print(f'eddy-permitting between {band.min():.0f} and '
          f'{band.max():.0f} deg (max L_d/dx = {ratio.max():.1f})')

DEPTH = 30.0

levels = uo_cube.coord(depth_coord_name(uo_cube)).points
index = int(np.argmin(np.abs(levels - DEPTH)))
constraint = iris.Constraint(
    **{depth_coord_name(uo_cube): levels[index]})
u_level = uo_cube.extract(constraint)
v_level = vo_cube.extract(constraint)
print(f'using level {levels[index]:.1f} m')

zeta = relative_vorticity(masked_data(u_level),
                          masked_data(v_level), u_level)

lat = lat_1d(u_level)
f = coriolis(lat)[:, None]
with np.errstate(divide='ignore', invalid='ignore'):
    rossby = zeta / f
rossby = np.ma.masked_where(
    np.broadcast_to(np.abs(lat)[:, None] < 2.0, rossby.shape),
    rossby)

## Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 5))
scale = np.ma.max(np.ma.abs(zeta))

mesh = axes[0].pcolormesh(lon_1d(u_level), lat, zeta,
                          cmap='RdBu_r', shading='auto',
                          vmin=-scale, vmax=scale)
axes[0].set_title(f'Relative vorticity at {levels[index]:.0f} m')
fig.colorbar(mesh, ax=axes[0], shrink=0.8, label='s$^{-1}$')

mesh = axes[1].pcolormesh(lon_1d(u_level), lat, rossby,
                          cmap='RdBu_r', shading='auto',
                          vmin=-0.5, vmax=0.5)
axes[1].set_title(r'Rossby number $\zeta/f$')
fig.colorbar(mesh, ax=axes[1], shrink=0.8, label='dimensionless')

for ax in axes:
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

## Shut the cluster down

Workers hold their memory until the cluster closes. Leaving one running
in an idle kernel is the usual reason a later notebook cannot get the
memory it asks for.

In [ ]:
client.close()